In [1]:
import os
from dotenv import load_dotenv
from datetime import date, timedelta
import requests
import zipfile
import io
import logging
from datetime import date, timedelta
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from edinet_xbrl.edinet_xbrl_parser import EdinetXbrlParser

load_dotenv()
EDINET_API_KEY = os.getenv('EDINET_API_KEY')

In [2]:
def get_documents_by_date(target_date, doc_type='030000'):
    """
    指定した日付にEDINETで開示された書類一覧を取得し、
    指定doc_type(有報)を満たすdoc_idとedinet_codeのリストを返す。
    """
    if isinstance(target_date, date):
        date_str = target_date.strftime("%Y-%m-%d")
    else:
        date_str = target_date
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents.json"
    params = {'date': date_str, 'type': 2,'Subscription-Key':EDINET_API_KEY}
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    results = data.get('results', [])

    docs = []


    for d in results:
        if d.get('formCode') == doc_type and d.get('docTypeCode') == '120':
            docs.append(d)

    return docs

h = get_documents_by_date(date(2024, 6, 11),doc_type='030000')

In [42]:
h

[{'seqNumber': 132,
  'docID': 'S100TKQK',
  'edinetCode': 'E03575',
  'secCode': '83660',
  'JCN': '6160001000993',
  'filerName': '株式会社滋賀銀行',
  'fundCode': None,
  'ordinanceCode': '010',
  'formCode': '030000',
  'docTypeCode': '120',
  'periodStart': '2023-04-01',
  'periodEnd': '2024-03-31',
  'submitDateTime': '2024-06-11 10:05',
  'docDescription': '有価証券報告書－第137期(2023/04/01－2024/03/31)',
  'issuerEdinetCode': None,
  'subjectEdinetCode': None,
  'subsidiaryEdinetCode': None,
  'currentReportReason': None,
  'parentDocID': None,
  'opeDateTime': None,
  'withdrawalStatus': '0',
  'docInfoEditStatus': '0',
  'disclosureStatus': '0',
  'xbrlFlag': '1',
  'pdfFlag': '1',
  'attachDocFlag': '1',
  'englishDocFlag': '0',
  'csvFlag': '1',
  'legalStatus': '1'}]

In [52]:
doc_id = h[0]['docID']
periodEnd_date= date.fromisoformat(h[0]['periodEnd'])

In [5]:
import os
import io
import zipfile
import tempfile
import requests
import logging

def download_xbrl_file(doc_id,output_dir = "./xbrl/"):
    """

    """
    path = output_dir + doc_id + '/'
    
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents"
    params = {
        'type': 1,
        'Subscription-Key': EDINET_API_KEY
    }
    
    try:
        response = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
        response.raise_for_status()
    except Exception as e:
        logging.error("EDINET APIからのダウンロードに失敗しました: %s", e)
        return None
    
    # try:
    # ダウンロードしたZIPアーカイブをメモリ上で読み込む
    
    z = zipfile.ZipFile(io.BytesIO(response.content))
    # 出力先ディレクトリの作成
    if not os.path.exists(path):
        os.makedirs(path)
            
    filename = doc_id + ".zip"
    with open(path+filename, 'wb') as f:    
        for chunk in response.iter_content(chunk_size=1024):
          f.write(chunk)

    with zipfile.ZipFile(path+filename) as zip_f:
        zip_f.extractall(path)


        # ZIP 内で拡張子が .xbrl のファイルを検索（大文字小文字区別しない）
        xbrl_files = [f for f in zip_f.namelist() if f.lower().endswith('.xbrl')]
        if not xbrl_files:
            logging.warning("doc_id=%s のZIP内にXBRLファイルが見つかりませんでした", doc_id)
            return None
        
        # 例として最初に見つかった XBRL ファイルの絶対パスを返す
        xbrl_file_rel_path = xbrl_files[0]
        current_directory = os.getcwd()
        xbrl_file_abs_path = os.path.join(current_directory + path[1:], xbrl_file_rel_path)
        
    return xbrl_file_abs_path
    

In [6]:
xbrl_file_abs_path = download_xbrl_file(doc_id)

In [7]:
xbrl_file_abs_path

'/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis/xbrl/S100TKQK/XBRL/PublicDoc/jpcrp030000-asr-001_E03575-000_2024-03-31_01_2024-06-11.xbrl'

In [224]:
import sys
import os
import io
import zipfile
import tempfile
import logging
import requests
from arelle import Cntlr

# ※ご自身の EDINET API キーを設定してください
EDINET_API_KEY = "YOUR_API_KEY_HERE"

def extract_fact_by_keyword(modelXbrl, keyword, 
                              role='http://disclosure.edinet-fsa.go.jp/role/jppfs/rol_BalanceSheet', 
                              lang='ja'):
    """
    modelXbrl 内の各 fact について、指定した role・言語で取得される概念ラベルに
    キーワード（例：「売上高」「配当政策」）が含まれていれば、その fact の値を返す。
    複数該当する場合は最初のものを返すシンプルな実装例です。
    """
    for fact in modelXbrl.facts:
        try:
            label = fact.concept.label(modelXbrl, role, lang)
        except Exception as e:
            label = ""
        if label and (keyword in label):
            return fact.value
    return None

def main(xbrl_path):
    """
    Arelle を用いて指定された XBRL ファイルから「売上高」と「配当政策」を抽出する。
    """
    # Arelle のコントローラを初期化（ログ出力先は 'logToPrint' に設定）
    cntlr = Cntlr.Cntlr(logFileName='logToPrint')
    
    # XBRL インスタンスファイルをロード
    modelXbrl = cntlr.modelManager.load(xbrl_path)
    if modelXbrl is None:
        print("XBRLファイルの読み込みに失敗しました。")
        sys.exit(1)
    
    # 売上高（Sales）の抽出
    sales_value = extract_fact_by_keyword(modelXbrl, "売上高")
    # 配当政策（Dividend Policy）の抽出
    dividend_policy_value = extract_fact_by_keyword(modelXbrl, "配当政策")
    
    print("【抽出結果】")
    print("売上高    : ", sales_value)
    print("配当政策  : ", dividend_policy_value)
    
    # 使用後はリソースを解放
    modelXbrl.close()

if __name__ == "__main__":

    main(xbrl_file_abs_path)

【抽出結果】
売上高    :  None
配当政策  :  None


In [8]:
## project.py

from arelle import Cntlr
import pandas as pd
from arelle import Cntlr
import glob




ctrl = Cntlr.Cntlr(logFileName='logToPrint')
model_xbrl = ctrl.modelManager.load(xbrl_file_abs_path)

fact_datas = list()

for fact in model_xbrl.facts:

   if fact.unit is not None and str(fact.unit.value) == 'JPY':

       
       label_ja = fact.concept.label(preferredLabel=None, lang='ja', linkroleHint=None)             
       x_value = fact.xValue

       if fact.context.startDatetime:
           start_date = fact.context.startDatetime
       else:
           start_date = None

       if fact.context.endDatetime:
           end_date = fact.context.endDatetime
       else:
           end_date = None

       fact_datas.append([
           label_ja,
           x_value,
           start_date,
           end_date,
           fact.contextID,
       ])
   else:
       continue

df = pd.DataFrame(fact_datas, 
                 columns=['勘定科目', '金額', '期首', '期末', 'contextID',] )

df_d = df[df['contextID'] == 'CurrentYearDuration']
df_i = df[df['contextID'] == 'CurrentYearInstant']

df_m = pd.concat([df_d, df_i])

In [16]:
from arelle import Cntlr
import pandas as pd
from arelle import Cntlr
import glob

ctrl = Cntlr.Cntlr(logFileName='logToPrint')
model_xbrl = ctrl.modelManager.load(xbrl_file_abs_path)

fact_datas = list()

In [23]:
model_xbrl.facts[0].concept.label(preferredLabel=None, lang='ja', linkroleHint=None) 

'提出回数'

In [67]:
import os
import pandas as pd
from arelle import Cntlr

def extract_financial_data(xbrl_file):
    # Arelle のコントローラ作成（ログ出力は標準出力に設定）
    cntlr = Cntlr.Cntlr(logFileName='logToPrint')
    
    # XBRL ファイルを読み込み
    modelXbrl = cntlr.modelManager.load(xbrl_file)
    
    # 抽出データを格納するリスト
    data = []
    
    # 各 fact から情報を抽出
    for fact in modelXbrl.facts:
        # 必要な情報例: コンセプト、値、単位、コンテキストID
        row = {
            'concept': fact.concept.qname.localName,
            'concept_jp':fact.concept.label(preferredLabel=None, lang='ja', linkroleHint=None), 
            'value': fact.value,
            'unit': fact.unitID if fact.unitID else '',
            'context': fact.contextID,

        }
        data.append(row)
    
    # リストを pandas DataFrame に変換
    df = pd.DataFrame(data)
    return df

def get_context_YYYY(context_str):
    """
    context の文字列に含まれるキーワードに基づいて、
    対象期を periodEnd から何年前か計算し、'YYYY' 形式で返す。
    """
    if "CurrentYear" in context_str:
        target_date = periodEnd_date
    elif "Prior1Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 1)
    elif "Prior2Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 2)
    elif "Prior3Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 3)
    elif "Prior4Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 4)
    else:
        # 対象外の場合は None を返す
        return None
    
    return target_date.strftime("%Y")


xbrl_path = xbrl_file_abs_path  # 解析対象の XBRL ファイルパスに変更
if os.path.exists(xbrl_path):
    df11 = extract_financial_data(xbrl_path)

else:
    print("XBRL ファイルが見つかりません:", xbrl_path)

In [68]:
assets = df11[df11['unit']=='JPY']

In [69]:
assets['context_YYYY'] = assets['context'].apply(get_context_YYYY)

/var/folders/0b/kvbdjp6d6kl4v5f97xnk5h480000gn/T/ipykernel_91799/352878298.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  assets['context_YYYY'] = assets['context'].apply(get_context_YYYY)


In [70]:
assets

,concept,concept_jp,value,unit,context,context_YYYY
1,OrdinaryIncomeSummaryOfBusinessResults,経常収益,88871000000,JPY,Prior4YearDuration,2020
2,OrdinaryIncomeSummaryOfBusinessResults,経常収益,85715000000,JPY,Prior3YearDuration,2021
3,OrdinaryIncomeSummaryOfBusinessResults,経常収益,98306000000,JPY,Prior2YearDuration,2022
4,OrdinaryIncomeSummaryOfBusinessResults,経常収益,115289000000,JPY,Prior1YearDuration,2023
5,OrdinaryIncomeSummaryOfBusinessResults,経常収益,122630000000,JPY,CurrentYearDuration,2024
...,...,...,...,...,...,...
1610,NetAssets,純資産,131167000000,JPY,CurrentYearInstant_NonConsolidatedMember_Valua...,2024
1611,NetAssets,純資産,30145000000,JPY,CurrentYearInstant_NonConsolidatedMember_Defer...,2024
1612,NetAssets,純資産,8240000000,JPY,CurrentYearInstant_NonConsolidatedMember_Reval...,2024
1613,NetAssets,純資産,169552000000,JPY,CurrentYearInstant_NonConsolidatedMember_Valua...,2024


In [45]:
for i in assets['context']:
    print(i)

Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearInstant
Prior3YearInstant
Prior2YearInstant
Prior1YearInstant
CurrentYearInstant
Prior4YearInstant
Prior3YearInstant
Prior2YearInstant
Prior1YearInstant
CurrentYearInstant
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearDuration
Prior3YearDuration
Prior2YearDuration
Prior1YearDuration
CurrentYearDuration
Prior4YearInstant
Prior3YearInstant
Prior2YearInst

In [41]:
df11

,concept,concept_jp,value,unit,context
0,NumberOfSubmissionDEI,提出回数,1,pure,FilingDateInstant
1,OrdinaryIncomeSummaryOfBusinessResults,経常収益,88871000000,JPY,Prior4YearDuration
2,OrdinaryIncomeSummaryOfBusinessResults,経常収益,85715000000,JPY,Prior3YearDuration
3,OrdinaryIncomeSummaryOfBusinessResults,経常収益,98306000000,JPY,Prior2YearDuration
4,OrdinaryIncomeSummaryOfBusinessResults,経常収益,115289000000,JPY,Prior1YearDuration
...,...,...,...,...,...
2248,OtherInformationFinancialStatementsEtcTextBlock,その他,"<p class=""smt_head3"" style=""orphans:0;widows:0...",,CurrentYearDuration
2249,OverviewOfOperationalProceduresForSharesTextBlock,提出会社の株式事務の概要,"\n<h2 class=""smt_head1"">第６ 【提出会社の株式事務の概要】</h2>...",,FilingDateInstant
2250,InformationAboutParentCompanyEtcOfReportingCom...,提出会社の親会社等の情報,\n１ 【提出会社の親会社等の情報】当行は、法第24条の７第１項に規定する親会社等はございま...,,FilingDateInstant
2251,OtherReferenceInformationTextBlock,その他の参考情報,"\n<h3 class=""smt_head2"" style=""font-family:&ap...",,FilingDateInstant


In [ ]:
df11[df11['unit']=='JPY']